# Luika Daily Drilling Report — PDF Agent

**Yaya Resources / Shanta New Luika. Rig DR10.**

This notebook takes Hamza's daily PDF and returns a CEO-ready update plus a list of
things that do not add up. Run it top to bottom the first time so you can see each
step; after that, jump to **Section 10** and point it at a folder.

### The one design rule

> **Python does the arithmetic. Claude does the language.**

The model never adds, divides or judges a number. It receives already-computed
fields and an already-computed exception list, and writes prose. A language model
asked to "check the numbers" will occasionally hand you a confident wrong figure
that *you* then have to defend in front of the Chairman. Arithmetic that must be
defensible belongs in code you can point at.

### What each section does

| § | Step | Output |
|---|---|---|
| 1 | Setup | libraries, and the API key read from `.env` |
| 2 | Config | rate card, plan, thresholds — **the only place you edit numbers** |
| 3 | Read | raw text off the PDF |
| 4–6 | Parse | header, shift block, week-to-date block |
| 7 | Derive | m/hr, L/hr, L/m, day totals |
| 8 | Validate | rules R1–R11 |
| 9 | **Money check** | drilled value vs invoiced value, by depth band |
| 10 | History | one row per day, trend chart |
| 11 | Narrate | Claude writes the update |
| 12 | Batch | run a whole folder |

## 1. Setup

In [1]:
# Run once. In VS Code: pick the kernel top-right, then Shift+Enter through the notebook.
# %pip install pdfplumber anthropic pandas matplotlib python-dotenv
import os, re, json, csv, sys
from pathlib import Path
from datetime import datetime, timedelta

import pdfplumber
import pandas as pd
import matplotlib.pyplot as plt

# Reads ANTHROPIC_API_KEY from a .env file sitting next to this notebook.
# Never paste the key into a cell - you will end up emailing this file to someone.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

ENCODING = "utf-8"      # everything is read and written as UTF-8, on every OS

pd.set_option("display.width", 140)
print("ready |", sys.version.split()[0],
      "| API key:", "found" if os.environ.get("ANTHROPIC_API_KEY") else "NOT SET")

ready | 3.12.3 | API key: NOT SET


## 2. Configuration

**Everything you might need to change lives here.** The rate card and the monthly
plan are `[given]` — they come from the `TAX INVOICE` and `PLANNED DRIL` sheets of
the August 2026 master workbook. Update them when the contract or the month changes,
and nothing downstream needs touching.

In [2]:
# --- Schedule of rates, TZS per metre  [given: TAX INVOICE!G23:G32] -------------
RATE_CARD = {
    ("NQ", "000-100m"): 351_000, ("NQ", "101-250m"): 377_000,
    ("NQ", "251-400m"): 416_000, ("NQ", "401-500m"): 442_000,
    ("NQ", "501-750m"): 462_800,
    ("PQ", "000-100m"): 377_000, ("PQ", "101-250m"): 403_000,
    ("HQ", "000-100m"): 377_000, ("HQ", "101-250m"): 403_000,
    ("HQ", "251-400m"): 429_000,
}
VAT_RATE = 0.18

# --- Monthly plan  [given: PLANNED DRIL!AH25 and TAX INVOICE!Y38] ---------------
MONTH_PLAN_METRES = 1_000
MONTH_PLAN_VALUE  = 377_780_000          # TZS, ex-VAT

# --- Fuel benchmark  [given: MASTAR 4,016 L / 800 m, August 2026] --------------
MONTH_L_PER_M = 5.02          # month-to-date litres per metre. Recompute monthly.

# --- Tolerances ----------------------------------------------------------------
RIG_HOUR_TOLERANCE   = 0.25        # hours a shift may exceed its own clock window
MONEY_FLAG_THRESHOLD = 5_000_000   # TZS gap worth raising
FUEL_OUTLIER_FACTOR  = 3           # x MONTH_L_PER_M before a shift is flagged
MAX_SHIFT_HOURS      = 13          # a stated shift window longer than this is wrong

# --- Files ---------------------------------------------------------------------
INBOX   = Path("inbox")      # drop Hamza's PDFs here
OUTDIR  = Path("out")        # updates land here
HISTORY = Path("history.csv")
OUTDIR.mkdir(exist_ok=True); INBOX.mkdir(exist_ok=True)

MODEL = "claude-sonnet-5"

BANDS = [(c, d) for c in ("NQ", "PQ", "HQ")
         for d in ("000-100m", "101-250m", "251-400m", "401-500m", "501-750m")
         if (c, d) in RATE_CARD]
print(f"{len(RATE_CARD)} rates loaded | plan {MONTH_PLAN_METRES} m "
      f"= TZS {MONTH_PLAN_VALUE:,}")

10 rates loaded | plan 1000 m = TZS 377,780,000


## 3. Read the PDF

The report is an Excel sheet printed to PDF, so the cell grid is gone and we are
working with text in reading order. That is workable but brittle — every figure the
notebook produces from a PDF gets tagged so you never forget where it came from.

**Change `PDF` below to whichever report you want to walk through.**

In [3]:
PDF = Path("inbox/Daily Report29.pdf")   # <-- edit me

def read_pdf_text(path: Path) -> str:
    with pdfplumber.open(path) as pdf:
        return "\n".join((p.extract_text() or "") for p in pdf.pages)

raw = read_pdf_text(PDF)
print(raw[:1400])

Invoice Total #############
Goal Chassing 85%
EXECUTIVE SUMMARY - LUIKA DRILLING PROGRAM
29-Aug-26
DAY SHIFT NIGHT SHIFT
Rig No. DR10 Rig No. DR10
Start Time 07:45 Start Time 19:45
End Shift 17:00 End Shift 04:00
Start Rig Hours 1816.0 Start Rig Hours 1826.0
End Rig Hours 1806.0 End Rig Hours 1833.0
Start Meter 144 Start Meter 177
End Meter 177 End Meter 195
Meter Drilled 33 Meter Drilled 18
Core Type NQ Core Type NQ
Refueling Time Refueling Time
Fuel Liters 104 Fuel Liters
Driller Name ALPHONCE MUNJORI Driller Name SADI KIONDO
Client Supervisor JUSTER Client Supervisor
Location GEPU Location GEPU
Hole ID GPDD 048 Hole ID GPDD 048
DR10 Performance
Actual(m) Plan(m)
70
60
50
40
30
20
10
0
BD1 BD2 BD3 BD4 BD5 BD6 BD7 BD8 BD9BD10BD11BD12BD13BD14BD15BD16BD17BD18BD19BD20BD21BD22BD23BD24BD25BD26BD27BD28BD29BD30BD31
PERFOMANCE EFFICIENCY
W E E K T O D A T E (WTD) P E R F O M A N C E (m)
MACHINE CORE BD29 BD30 BD31 - - - - -
000-100m NQ - - - 1 27.00 - - - -
101-250m NQ 41.00 - - 5 21.00 - - -

## 4. Parse the header

The date, and the `Goal Chassing` percentage. That percentage matters more than it
looks: it is the only place on the printout where the **invoice value** is visible,
because the `Invoice Total` cell is too narrow and renders as `#########`. Section 9
uses it to back out what has actually been billed.

In [4]:
MONTHS = "Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split()

def parse_header(text: str) -> dict:
    h = {"report_date": None, "goal_chasing": None, "business_day": None}

    m = re.search(r"(\d{1,2})-([A-Za-z]{3})-(\d{2,4})", text)
    if m:
        d, mon, y = int(m.group(1)), m.group(2).title(), int(m.group(3))
        y += 2000 if y < 100 else 0
        h["report_date"] = datetime(y, MONTHS.index(mon) + 1, d).date()
        h["business_day"] = f"BD{d}"          # the sheet numbers BD by day of month

    g = re.search(r"Goal\s+Chas+ing\s+([\d.]+)\s*%", text, re.I)
    if g:
        h["goal_chasing"] = float(g.group(1)) / 100
    return h

header = parse_header(raw)
header

{'report_date': datetime.date(2026, 8, 29),
 'goal_chasing': 0.85,
 'business_day': 'BD29'}

## 5. Parse the shift block

The template prints each label twice on one line — day value first, night value
second:

```
Start Rig Hours 1816.0 Start Rig Hours 1826.0
```

So we split on the label itself and keep the non-empty segments in order. That is
more robust than counting tokens, which breaks the moment a driller has two names.

In [5]:
LABELS = {
    "rig_no": "Rig No.", "start_time": "Start Time", "end_shift": "End Shift",
    "start_rig_hours": "Start Rig Hours", "end_rig_hours": "End Rig Hours",
    "start_meter": "Start Meter", "end_meter": "End Meter",
    "meter_drilled": "Meter Drilled", "core_type": "Core Type",
    "fuel_litres": "Fuel Liters", "driller": "Driller Name",
    "client_supervisor": "Client Supervisor", "location": "Location",
    "hole_id": "Hole ID",
}
NUMERIC = {"start_rig_hours", "end_rig_hours", "start_meter",
           "end_meter", "meter_drilled", "fuel_litres"}

def to_num(v):
    if v is None: return None
    s = str(v).replace(",", "").replace("(", "-").replace(")", "").strip()
    if s in {"", "-", "\u2014"}: return None
    try: return float(s)
    except ValueError: return None

def parse_shifts(text: str) -> dict:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    shifts = {"day": {"shift": "day"}, "night": {"shift": "night"}}

    for field, label in LABELS.items():
        segs = []
        for l in lines:
            if l.lower().startswith(label.lower()):
                segs = [s.strip() for s in re.split(re.escape(label), l, flags=re.I)]
                segs = [s for s in segs if s]
                break
        for name, seg in zip(("day", "night"), segs):
            shifts[name][field] = to_num(seg) if field in NUMERIC else seg
        for name in ("day", "night"):                 # never leave a key missing
            shifts[name].setdefault(field, None)
    return shifts

shifts = parse_shifts(raw)
pd.DataFrame(shifts).T

,shift,rig_no,start_time,end_shift,start_rig_hours,end_rig_hours,start_meter,end_meter,meter_drilled,core_type,fuel_litres,driller,client_supervisor,location,hole_id
day,day,DR10,07:45,17:00,1816.0,1806.0,144.0,177.0,33.0,NQ,104.0,ALPHONCE MUNJORI,JUSTER,GEPU,GPDD 048
night,night,DR10,19:45,04:00,1826.0,1833.0,177.0,195.0,18.0,NQ,None,SADI KIONDO,None,GEPU,GPDD 048


### The operations note

One free-text box, and currently the **only** place a machine problem or a missing
consumable ever appears. On 29 August it is the only record anywhere that three
hours went on a worn-out drill bit.

In [6]:
def parse_ops_note(text: str) -> str:
    m = re.search(r"OPERAT\w*\s*UPDATE:?\s*(.+?)(?:\n\s*\n|W E E K|$)",
                  text, re.S | re.I)
    return " ".join(m.group(1).split()) if m else ""

ops_note = parse_ops_note(raw)
print(ops_note or "(blank)")

N/S --- The night shift encountered the challenge of the drill bit running out.which took them about 3 hours to pull the drill rod out to replace the bit and return it to the hole to continue drilling


## 6. Parse the week-to-date block

This is the table that feeds the invoice, so it is the one worth getting right.

Two quirks of the PDF, both handled below:

1. **Split numbers.** `127.00` comes out of the extractor as `1` then `27.00`;
   `6.00` comes out as `6` then `.00`. We stitch a bare 1–3 digit token onto a
   following decimal token.
2. **The month-total column.** In the last week of the month the sheet's own
   column formula overruns the calendar and starts pulling the monthly totals
   column. The header for it is blank, so any column whose header is not a `BDnn`
   label but which carries values is treated as the month total. That is not a
   feature of the sheet — it is a side effect — but it is useful, because it puts
   the whole month's metres on the daily printout.

In [7]:
BAND_LABELS = [(f"{d} {c}", (c, d)) for (c, d) in RATE_CARD]

def _repair(tokens):
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if re.fullmatch(r"\d{1,3}", t) and i + 1 < len(tokens) \
           and re.fullmatch(r"[\d,]*\.\d+", tokens[i + 1]):
            out.append((t + tokens[i + 1]).replace(",", "")); i += 2
        else:
            out.append(t.replace(",", "")); i += 1
    return out

def _cells(tokens):
    vals = []
    for t in _repair(tokens):
        if t in {"-", "\u2014"}:            vals.append(None)
        elif re.fullmatch(r"-?\d+(\.\d+)?", t): vals.append(float(t))
    return vals

def parse_wtd(text: str) -> dict:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    headers, rows = [], {}
    for l in lines:
        if l.startswith("MACHINE CORE"):
            headers = l.replace("MACHINE CORE", "").split()
        for prefix, key in BAND_LABELS:
            if l.startswith(prefix + " "):
                rows[key] = _cells(l[len(prefix):].split())

    n = len(headers)
    for k, v in rows.items():
        rows[k] = (v + [None] * n)[:n]

    total_col = next(
        (i for i, h in enumerate(headers)
         if not re.fullmatch(r"BD\d+", h)
         and any((rows.get(k) or [None] * n)[i] for k in rows)),
        None)
    return {"headers": headers, "rows": rows, "total_col": total_col}

wtd = parse_wtd(raw)
wtd_df = pd.DataFrame(
    {f"{c} {d}": wtd["rows"].get((c, d)) for (c, d) in RATE_CARD if (c, d) in wtd["rows"]},
    index=[h if h != "-" else "MONTH" for h in wtd["headers"]],
).T
print("month-total column:", wtd["total_col"])
wtd_df

month-total column: 3


,BD29,BD30,BD31,MONTH,MONTH,MONTH,MONTH,MONTH
NQ 000-100m,NaN,NaN,NaN,127.0,NaN,NaN,NaN,NaN
NQ 101-250m,41.0,NaN,NaN,521.0,NaN,NaN,NaN,NaN
NQ 251-400m,NaN,NaN,NaN,194.0,NaN,NaN,NaN,NaN
HQ 000-100m,NaN,NaN,NaN,154.0,NaN,NaN,NaN,NaN
HQ 101-250m,None,None,None,None,None,None,None,None


## 7. Derive the metrics

Nothing here is opinion. Rig hours are the hour-meter difference, metres are the
counter difference, and the rates follow.

**Rate of penetration (m/hr) is the drilling equivalent of cost per operating hour.**
Metres alone tell you nothing — 30 m in 18 hours and 30 m in 9 hours are different
businesses. Always divide by *rig hours actually turning*, never by shift length: if
you use shift length you bury the downtime inside the rate, and a spares problem
looks like a drilling-speed problem. Two very different conversations with Hamza.

In [8]:
def clock_hours(start, end):
    try:
        sh, sm = [int(x) for x in re.split(r"[:.]", str(start))[:2]]
        eh, em = [int(x) for x in re.split(r"[:.]", str(end))[:2]]
    except Exception:
        return None
    s, e = sh * 60 + sm, eh * 60 + em
    if e <= s: e += 24 * 60           # crosses midnight
    return round((e - s) / 60, 2)

def derive(shifts: dict) -> dict:
    for s in shifts.values():
        s["rig_hours"] = (round(s["end_rig_hours"] - s["start_rig_hours"], 2)
                          if None not in (s["start_rig_hours"], s["end_rig_hours"]) else None)
        s["metres"] = (round(s["end_meter"] - s["start_meter"], 2)
                       if None not in (s["start_meter"], s["end_meter"]) else None)
        s["scheduled_hours"] = clock_hours(s["start_time"], s["end_shift"])
        rh, m, f = s["rig_hours"], s["metres"], s["fuel_litres"]
        s["m_per_hr"]  = round(m / rh, 2) if (rh and rh > 0 and m is not None) else None
        s["l_per_hr"]  = round(f / rh, 2) if (rh and rh > 0 and f) else None
        s["l_per_m"]   = round(f / m, 2)  if (m and m > 0 and f) else None
    return shifts

shifts = derive(shifts)

def day_totals(shifts):
    rh = [s["rig_hours"] for s in shifts.values() if s["rig_hours"] is not None]
    m  = [s["metres"]    for s in shifts.values() if s["metres"] is not None]
    f  = [s["fuel_litres"] for s in shifts.values() if s["fuel_litres"] is not None]
    t = {"rig_hours": round(sum(rh), 2) if rh else None,
         "metres": round(sum(m), 2) if m else None,
         "fuel_litres": round(sum(f), 2) if f else None}
    t["m_per_hr"] = (round(t["metres"] / t["rig_hours"], 2)
                     if t["rig_hours"] and t["rig_hours"] > 0 and t["metres"] is not None else None)
    return t

totals = day_totals(shifts)
display(pd.DataFrame(shifts).T[["rig_hours","scheduled_hours","metres",
                                "m_per_hr","fuel_litres","l_per_hr","l_per_m"]])
totals

,rig_hours,scheduled_hours,metres,m_per_hr,fuel_litres,l_per_hr,l_per_m
day,-10.0,9.25,33.0,None,104.0,None,3.15
night,7.0,8.25,18.0,2.57,None,None,None


{'rig_hours': -3.0, 'metres': 51.0, 'fuel_litres': 104.0, 'm_per_hr': None}

## 8. Validation rules

Eleven rules. Each one exists because it has already caught something real in the
August reports. Severity drives what you do with it: `CRITICAL` means stop, `HIGH`
means resolve before the invoice, `MEDIUM` means chase it with Hamza tomorrow.

| Rule | Checks | Caught |
|---|---|---|
| R1 | metre counter matches the "Meter Drilled" box | |
| R2 | rig hours positive and within the shift window | 29 Aug day shift runs **backwards** (1816 → 1806); night shift 10 h inside an 8.25 h window on 25 and 26 Aug |
| R3 | fuel recorded on both shifts | night fuel blank on 26, 29 and 30 Aug |
| R4 | hole ID, core type, driller, location present | |
| R5 | hole ID changes when the counter resets to zero | 26 Aug started a new hole still labelled GPDD 048 |
| R6 | shift-to-shift continuity inside the day | |
| R7 | no rig-hour gap or reversal against yesterday | 5 h unaccounted between 25 and 26 Aug; meter goes backwards 7 h on 30 Aug |
| R8 | **WTD figure matches the shift log** | 29 Aug logs 51 m in the shifts, books 41 m in the table |
| R9 | core type in the shift log matches the band booked | 26 Aug shifts say NQ, table books HQ |
| R10 | today's column is inside the printed WTD window | |
| R11 | the operations note is not blank | |
| R12 | fuel per metre within 3x the month norm | 30 Aug day shift burned **38.8 L/m** against a 5.02 L/m norm — the rig idled waiting for water |
| R13 | the stated shift window is a plausible length | 30 Aug day shift printed as 07:45–02:30, an 18.75 h "day" |
| R14 | day totals suppressed when R7 has fired | 30 Aug day total of 0.46 m/hr is contaminated by a copied-forward night column |
| R15 | a stoppage in the free-text note that no field counts | 29 Aug lost ~3 h to a worn bit; 30 Aug lost 6 h waiting for water |

In [9]:
def validate(header, shifts, totals, wtd, ops_note="", prev_row=None):
    ex = []
    def flag(code, sev, msg): ex.append({"code": code, "severity": sev, "message": msg})

    flag("SOURCE", "MEDIUM",
         "Parsed from PDF, not Excel. The cell grid is lost, so field mapping is "
         "positional. Ask Hamza for the .xlsx to remove this whole class of risk.")

    for name, s in shifts.items():
        N = name.upper()

        if None not in (s["metres"], s["meter_drilled"]) \
           and abs(s["metres"] - s["meter_drilled"]) > 0.01:
            flag("R1", "HIGH", f"{N}: counter says {s['metres']:.1f} m "
                 f"({s['start_meter']:g}\u2192{s['end_meter']:g}) but the "
                 f"'Meter Drilled' box says {s['meter_drilled']:.1f} m.")

        rh, sch = s["rig_hours"], s["scheduled_hours"]
        if rh is None:
            flag("R2", "HIGH", f"{N}: hour-meter readings missing. This shift adds "
                               f"nothing to the day's rig hours, so the day is understated.")
        elif rh < 0:
            flag("R2", "CRITICAL", f"{N}: rig hours are NEGATIVE ({rh:g}) \u2014 start "
                 f"{s['start_rig_hours']:g}, end {s['end_rig_hours']:g}. The readings are "
                 f"the wrong way round. No rate for this shift is defensible.")
        elif rh == 0:
            flag("R2", "HIGH", f"{N}: zero rig hours recorded.")
        elif sch and rh > sch + RIG_HOUR_TOLERANCE:
            flag("R2", "HIGH", f"{N}: rig ran {rh:.1f} h inside a {sch:.2f} h window "
                 f"({s['start_time']}\u2013{s['end_shift']}). Impossible as stated \u2014 either "
                 f"the shift times or the hour meter is wrong.")

        if s["fuel_litres"] is None:
            flag("R3", "MEDIUM", f"{N}: fuel litres blank. Cost per metre cannot be built.")

        for f_ in ("hole_id", "core_type", "driller", "location"):
            if not s.get(f_):
                flag("R4", "MEDIUM", f"{N}: '{f_}' is blank.")

    d, n = shifts["day"], shifts["night"]
    if d["start_meter"] == 0 and d.get("hole_id") and d["hole_id"] == n.get("hole_id"):
        flag("R5", "HIGH", f"Counter reset to 0 (new hole) but Hole ID is still "
             f"{d['hole_id']}. The finished hole's final depth and the new hole ID are lost.")

    if None not in (d["end_meter"], n["start_meter"]) \
       and d["end_meter"] != n["start_meter"] and n["start_meter"] != 0:
        flag("R6", "HIGH", f"Day ends at metre {d['end_meter']:g} but night starts at "
             f"{n['start_meter']:g}. The two shifts do not join up.")

    if prev_row is not None and prev_row.get("max_rig_hours"):
        prev_end = float(prev_row["max_rig_hours"])
        today = [x for s in shifts.values()
                 for x in (s["start_rig_hours"], s["end_rig_hours"]) if x is not None]
        if today:
            gap = round(min(today) - prev_end, 2)
            if gap > 0.1:
                flag("R7", "HIGH", f"{gap:.1f} rig hours unaccounted between the previous "
                     f"report (ended {prev_end:.1f}) and this one (starts {min(today):.1f}).")
            elif gap < -0.1:
                flag("R7", "HIGH", f"Hour meter goes BACKWARDS by {abs(gap):.1f} h versus "
                     f"the previous report. Either a shift column was copied forward "
                     f"unchanged or a reading was mistyped.")

    bd = header.get("business_day")
    if bd and bd in wtd["headers"]:
        i = wtd["headers"].index(bd)
        booked = sum(v[i] for v in wtd["rows"].values() if v[i])
        if totals["metres"] is not None and abs(booked - totals["metres"]) > 0.01:
            flag("R8", "CRITICAL", f"The WTD table books {booked:.1f} m for {bd} but the "
                 f"shift log totals {totals['metres']:.1f} m. The WTD table is what feeds "
                 f"the invoice \u2014 this is a billing exposure, not a typo.")
        booked_cores = {c for (c, _), v in wtd["rows"].items() if v[i]}
        log_cores = {s["core_type"] for s in shifts.values()
                     if s.get("core_type") and s.get("metres")}
        if booked_cores and log_cores and not (booked_cores & log_cores):
            flag("R9", "CRITICAL", f"Shift log says core {'/'.join(sorted(log_cores))} but "
                 f"the table books {bd} under {'/'.join(sorted(booked_cores))}. Core sizes "
                 f"bill at different rates.")
    elif bd:
        flag("R10", "MEDIUM", f"{bd} is not in the WTD window {wtd['headers']}, so today's "
             f"figure cannot be reconciled against the table on this printout.")

    if not ops_note:
        flag("R11", "MEDIUM", "No operations note. If the rig ran clean, say so \u2014 a blank "
                              "box is not the same as 'no issues'.")

    # R12 - fuel far above the month norm means the rig burned diesel without drilling.
    #       This is an idling problem, not a fuel-efficiency problem. Different fix.
    for name, s in shifts.items():
        if s["l_per_m"] and s["l_per_m"] > MONTH_L_PER_M * FUEL_OUTLIER_FACTOR:
            flag("R12", "HIGH", f"{name.upper()}: {s['l_per_m']:.1f} L/m against a "
                 f"{MONTH_L_PER_M} L/m month norm ({s['l_per_m'] / MONTH_L_PER_M:.1f}x). "
                 f"The rig burned fuel without drilling. Get the standby hours and the "
                 f"reason, and establish whether standby is billable.")

    # R13 - a stated shift window longer than a shift
    for name, s in shifts.items():
        if s["scheduled_hours"] and s["scheduled_hours"] > MAX_SHIFT_HOURS:
            flag("R13", "HIGH", f"{name.upper()}: window {s['start_time']}\u2013"
                 f"{s['end_shift']} is {s['scheduled_hours']:.2f} h. Either the times are "
                 f"wrong or somebody worked a double. Utilisation off this is meaningless.")

    # R14 - once a shift column is suspect, the DAY totals are contaminated
    if any(e["code"] == "R7" for e in ex):
        flag("R14", "HIGH", "R7 fired, so one shift column is suspect. Do not quote the "
                            "day total or the day m/hr from this report \u2014 use the clean "
                            "shift on its own and say which one.")

    # R15 - a standby or breakdown described in free text that no field captures
    if ops_note and re.search(r"standby|stand by|break\s?down|waiting|no production|"
                              r"bit|water|spare", ops_note, re.I):
        flag("R15", "HIGH", f"The operations note describes a stoppage that no field on "
             f"this report counts: \"{ops_note[:160]}\". Downtime hours and consumables "
             f"are narrative here, not data. Nothing totals them across the month.")
    return ex

exceptions = validate(header, shifts, totals, wtd, ops_note)
pd.DataFrame(exceptions)

,code,severity,message
0,SOURCE,MEDIUM,"Parsed from PDF, not Excel. The cell grid is l..."
1,R2,CRITICAL,DAY: rig hours are NEGATIVE (-10) — start 1816...
2,R3,MEDIUM,NIGHT: fuel litres blank. Cost per metre canno...
3,R8,CRITICAL,The WTD table books 41.0 m for BD29 but the sh...
4,R15,HIGH,The operations note describes a stoppage that ...


## 9. The money check

This is the section that pays for the notebook.

Revenue on this contract is `metres x rate`, band by band. The printout hides the
invoice total behind `#########`, but `Goal Chassing` is that same total divided by
the full-month plan value — so multiplying it back out recovers roughly what has
been billed. Compare that to the value of what was actually drilled, and any band
that is missing from the invoice shows up as a gap.

On the August 2026 workbook this is not hypothetical: the `HQ 000-100m` line of the
`TAX INVOICE` sheet reads `='ACTUAL DRIL'!AA18` when it should read
`='ACTUAL DRIL'!AH17`, so every metre of HQ drilling falls off the invoice.

**Watch the two attainment lines as well.** `Goal Chassing` divides *month-to-date*
value by the *full-month* plan, so it cannot reach 100% until the last day of the
month and tells you nothing about whether the rig is on plan today. Metres against
plan metres is the honest comparison.

In [10]:
def money_check(wtd, header, exceptions):
    i = wtd["total_col"]
    if i is None:
        print("No month-total column on this printout (it appears in the final week "
              "of the month). Money check skipped.")
        return None

    rows = []
    for (core, depth), vals in wtd["rows"].items():
        m = vals[i]
        if not m: continue
        rate = RATE_CARD[(core, depth)]
        rows.append({"core": core, "band": depth, "metres": m,
                     "rate": rate, "value": m * rate})
    df = pd.DataFrame(rows).sort_values("value", ascending=False)
    drilled_m, drilled_v = df["metres"].sum(), df["value"].sum()

    print(f"Month to date, from this printout  [derived]")
    print(f"  metres drilled : {drilled_m:,.0f} m  vs plan {MONTH_PLAN_METRES:,} m "
          f"= {drilled_m / MONTH_PLAN_METRES:.1%}")
    print(f"  value drilled  : TZS {drilled_v:,.0f} ex-VAT  vs plan "
          f"TZS {MONTH_PLAN_VALUE:,} = {drilled_v / MONTH_PLAN_VALUE:.1%}")

    g = header.get("goal_chasing")
    if g is None:
        print("\n  'Goal Chassing' not readable \u2014 cannot infer what was billed.")
        return df

    implied = g * MONTH_PLAN_VALUE
    gap = drilled_v - implied
    print(f"\n  'Goal Chassing' on the sheet : {g:.0%}")
    print(f"  implied invoiced value       : TZS {implied:,.0f}")
    print(f"  gap vs value drilled         : TZS {gap:,.0f}")

    if gap > MONEY_FLAG_THRESHOLD:
        near = df.iloc[(df["value"] - gap).abs().argsort()].iloc[0]
        exceptions.append({
            "code": "MONEY-1", "severity": "CRITICAL",
            "message": (
                f"TZS {gap:,.0f} of drilled work does not appear to be on the invoice. "
                f"The closest match is {near['core']} {near['band']}: {near['metres']:,.0f} m "
                f"at TZS {near['rate']:,}/m = TZS {near['value']:,.0f}. Check that the "
                f"invoice line for that band pulls its own band's monthly total. "
                f"Note the percentage is rounded on the printout, so treat this as "
                f"approximate until the .xlsx is checked."),
        })
        print(f"\n  >>> FLAGGED. Closest band: {near['core']} {near['band']} "
              f"= TZS {near['value']:,.0f}")
    else:
        print("\n  Invoiced value is consistent with what was drilled.")
    return df

bands_df = money_check(wtd, header, exceptions)
bands_df

Month to date, from this printout  [derived]
  metres drilled : 996 m  vs plan 1,000 m = 99.6%
  value drilled  : TZS 379,756,000 ex-VAT  vs plan TZS 377,780,000 = 100.5%

  'Goal Chassing' on the sheet : 85%
  implied invoiced value       : TZS 321,113,000
  gap vs value drilled         : TZS 58,643,000

  >>> FLAGGED. Closest band: HQ 000-100m = TZS 58,058,000


,core,band,metres,rate,value
1,NQ,101-250m,521.0,377000,196417000.0
2,NQ,251-400m,194.0,416000,80704000.0
3,HQ,000-100m,154.0,377000,58058000.0
0,NQ,000-100m,127.0,351000,44577000.0


## 10. History

The daily sheet in the master workbook holds **one day** and is overwritten every
morning — rig hours, fuel, hole ID and the operations note are destroyed daily.
That is why no availability figure exists anywhere in the business.

This CSV is the fix. It costs nothing and after a month it answers *"is DR10 getting
better or worse"*, which is a question about the asset rather than about yesterday.

In [11]:
FIELDS = ["report_date","business_day","rig_no","hole_id","metres","rig_hours",
          "m_per_hr","fuel_litres","max_rig_hours","goal_chasing","n_exceptions",
          "n_critical","source_file"]

def append_history(header, shifts, totals, exceptions, path):
    hours = [x for s in shifts.values()
             for x in (s["start_rig_hours"], s["end_rig_hours"]) if x is not None]
    row = {
        "report_date": header["report_date"], "business_day": header["business_day"],
        "rig_no": shifts["day"]["rig_no"], "hole_id": shifts["day"]["hole_id"],
        "metres": totals["metres"], "rig_hours": totals["rig_hours"],
        "m_per_hr": totals["m_per_hr"], "fuel_litres": totals["fuel_litres"],
        "max_rig_hours": max(hours) if hours else None,
        "goal_chasing": header["goal_chasing"],
        "n_exceptions": len(exceptions),
        "n_critical": sum(1 for e in exceptions if e["severity"] == "CRITICAL"),
        "source_file": path.name,
    }
    existing = []
    if HISTORY.exists():
        existing = [r for r in csv.DictReader(HISTORY.open(encoding=ENCODING))
                    if r["report_date"] != str(row["report_date"])]   # re-runs replace
    with HISTORY.open("w", newline="", encoding=ENCODING) as f:
        w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader()
        w.writerows(existing); w.writerow(row)
    return row

def previous_row(report_date):
    if not HISTORY.exists(): return None
    rows = sorted((r for r in csv.DictReader(HISTORY.open(encoding=ENCODING))
                   if r["report_date"]),
                  key=lambda r: r["report_date"])
    prior = [r for r in rows if r["report_date"] < str(report_date)]
    return prior[-1] if prior else None

append_history(header, shifts, totals, exceptions, PDF)
pd.read_csv(HISTORY)

,report_date,business_day,rig_no,hole_id,metres,rig_hours,m_per_hr,fuel_litres,max_rig_hours,goal_chasing,n_exceptions,n_critical,source_file
0,2026-08-29,BD29,DR10,GPDD 048,51.0,-3.0,NaN,104.0,1833.0,0.85,6,3,Daily Report29.pdf


In [12]:
def plot_history():
    if not HISTORY.exists(): return
    df = pd.read_csv(HISTORY).sort_values("report_date")
    if len(df) < 2:
        print("Need at least two days before a trend means anything."); return
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].bar(df["business_day"], df["metres"], color="#3b6ea5")
    ax[0].set_title("Metres per day"); ax[0].set_ylabel("m")
    ax[1].plot(df["business_day"], df["m_per_hr"], marker="o", color="#a53b3b")
    ax[1].set_title("Metres per rig hour"); ax[1].set_ylabel("m/hr")
    ax[1].axhline(0, lw=0.8, color="grey")
    for a in ax: a.tick_params(axis="x", rotation=45); a.grid(alpha=.25)
    plt.tight_layout(); plt.show()

plot_history()

Need at least two days before a trend means anything.


## 11. Let Claude write the update

Now, and only now, the model gets involved. It receives the computed fields and the
exception list as JSON and turns them into five short blocks. It is instructed not to
calculate anything and not to soften an exception.

Set your key first: `export ANTHROPIC_API_KEY=sk-...` before launching Jupyter, or
uncomment the line below.

In [13]:
# The key was loaded from .env back in Section 1. If this prints NOT SET, check that
# a file called .env sits beside this notebook containing:  ANTHROPIC_API_KEY=sk-ant-...
print("API key:", "found" if os.environ.get("ANTHROPIC_API_KEY") else "NOT SET")

SYSTEM = '''You write the daily drilling update for the CEO of Yaya Resources, a
Tanzanian mining contractor. The rig is on Shanta's New Luika site.

Absolute rules:
- Use ONLY the numbers in the JSON you are given. Do not add, subtract, average,
  convert or estimate any figure. If a number is not there, write "not recorded".
- Every exception in the list must appear. Do not soften one.
- No filler, no praise, no restating the question. Short sentences, plain English.
- If the exception list is empty, say the checks passed. Do not invent concerns.

Output exactly these five blocks and nothing else:

PRODUCTION - metres and rig hours by shift, and the day total.
MACHINE STATUS - what the report says about the rig running. If there is no fault
  log, service record or downtime reason on the sheet, say so plainly.
CONSUMABLES, TOOLS AND SPARES - report what is stated. If nothing is stated, write
  "Not covered in the report submitted."
ISSUES AND ACTIONS - the site's own operations note, then what it does not answer.
DATA INTEGRITY FLAGS - the exception list, one line each.

Under 220 words.'''

def build_payload(header, shifts, totals, wtd, ops_note, exceptions):
    return {
        "report_date": str(header["report_date"]),
        "business_day": header["business_day"],
        "goal_chasing_on_sheet": header["goal_chasing"],
        "shifts": shifts, "day_totals": totals,
        "operations_note": ops_note or None,
        "wtd_headers": wtd["headers"],
        "exceptions": exceptions,
    }

def narrate(payload, dry_run=False):
    text = json.dumps(payload, indent=2, default=str)
    if dry_run or not os.environ.get("ANTHROPIC_API_KEY"):
        return "[dry run \u2014 no API key set]\n\n" + text
    from anthropic import Anthropic
    msg = Anthropic().messages.create(
        model=MODEL, max_tokens=1200, system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Rig report data:\n\n{text}\n\nWrite the CEO update."}])
    return "".join(b.text for b in msg.content if b.type == "text")

payload = build_payload(header, shifts, totals, wtd, ops_note, exceptions)
update = narrate(payload, dry_run=True)      # flip to False once your key is set
print(update[:2500])

API key: NOT SET
[dry run — no API key set]

{
  "report_date": "2026-08-29",
  "business_day": "BD29",
  "goal_chasing_on_sheet": 0.85,
  "shifts": {
    "day": {
      "shift": "day",
      "rig_no": "DR10",
      "start_time": "07:45",
      "end_shift": "17:00",
      "start_rig_hours": 1816.0,
      "end_rig_hours": 1806.0,
      "start_meter": 144.0,
      "end_meter": 177.0,
      "meter_drilled": 33.0,
      "core_type": "NQ",
      "fuel_litres": 104.0,
      "driller": "ALPHONCE MUNJORI",
      "client_supervisor": "JUSTER",
      "location": "GEPU",
      "hole_id": "GPDD 048",
      "rig_hours": -10.0,
      "metres": 33.0,
      "scheduled_hours": 9.25,
      "m_per_hr": null,
      "l_per_hr": null,
      "l_per_m": 3.15
    },
    "night": {
      "shift": "night",
      "rig_no": "DR10",
      "start_time": "19:45",
      "end_shift": "04:00",
      "start_rig_hours": 1826.0,
      "end_rig_hours": 1833.0,
      "start_meter": 177.0,
      "end_meter": 195.0,
      "met

## 12. Batch run

Drop a folder of PDFs in `inbox/` and run this. It processes them **in date order**
so the day-over-day continuity rule (R7) has yesterday to compare against, writes one
markdown file per day into `out/`, and prints a summary.

This is the cell you use daily once you have followed the notebook through once.

In [14]:
def process(path: Path, dry_run=True):
    raw = read_pdf_text(path)
    header = parse_header(raw)
    shifts = derive(parse_shifts(raw))
    totals = day_totals(shifts)
    wtd    = parse_wtd(raw)
    note   = parse_ops_note(raw)

    ex = validate(header, shifts, totals, wtd, note,
                  previous_row(header["report_date"]))
    money_check(wtd, header, ex)

    append_history(header, shifts, totals, ex, path)
    payload = build_payload(header, shifts, totals, wtd, note, ex)
    body = narrate(payload, dry_run=dry_run)

    stamp = header["report_date"] or path.stem
    out = OUTDIR / f"{stamp}_ceo_update.md"
    out.write_text(
        f"# Luika drilling \u2014 {stamp}\n"
        f"Rig {shifts['day']['rig_no']} | source: {path.name} | "
        f"{len(ex)} flag(s), {sum(1 for e in ex if e['severity']=='CRITICAL')} critical\n\n"
        + body + "\n", encoding=ENCODING)
    return {"file": path.name, "date": header["report_date"],
            "metres": totals["metres"], "rig_hours": totals["rig_hours"],
            "flags": len(ex),
            "critical": sum(1 for e in ex if e["severity"] == "CRITICAL"),
            "high": sum(1 for e in ex if e["severity"] == "HIGH"),
            "exceptions": ex}

def run_folder(folder=INBOX, dry_run=True):
    pdfs = sorted(folder.glob("*.pdf"),
                  key=lambda p: parse_header(read_pdf_text(p))["report_date"]
                                or datetime.min.date())
    results = [process(p, dry_run) for p in pdfs]
    print("\n" + "=" * 70)
    display(pd.DataFrame(results)[["date","file","metres","rig_hours",
                                   "flags","critical","high"]])
    for r in results:
        loud = [e for e in r["exceptions"] if e["severity"] in ("CRITICAL", "HIGH")]
        if loud:
            print(f"\n{r['date']}")
            for e in loud:
                print(f"  [{e['severity']:8}] {e['code']:7} {e['message']}")
    return results

results = run_folder(dry_run=True)

No month-total column on this printout (it appears in the final week of the month). Money check skipped.


No month-total column on this printout (it appears in the final week of the month). Money check skipped.
Month to date, from this printout  [derived]
  metres drilled : 996 m  vs plan 1,000 m = 99.6%
  value drilled  : TZS 379,756,000 ex-VAT  vs plan TZS 377,780,000 = 100.5%

  'Goal Chassing' on the sheet : 85%
  implied invoiced value       : TZS 321,113,000
  gap vs value drilled         : TZS 58,643,000

  >>> FLAGGED. Closest band: HQ 000-100m = TZS 58,058,000


Month to date, from this printout  [derived]
  metres drilled : 1,002 m  vs plan 1,000 m = 100.2%
  value drilled  : TZS 382,018,000 ex-VAT  vs plan TZS 377,780,000 = 101.1%

  'Goal Chassing' on the sheet : 86%
  implied invoiced value       : TZS 324,890,800
  gap vs value drilled         : TZS 57,127,200

  >>> FLAGGED. Closest band: HQ 000-100m = TZS 58,058,000



,date,file,metres,rig_hours,flags,critical,high
0,2026-08-25,Daily_Report25.pdf,30.0,18.0,4,1,2
1,2026-08-26,Daily_Report26.pdf,33.0,17.0,8,2,4
2,2026-08-29,Daily Report29.pdf,51.0,-3.0,8,3,3
3,2026-08-30,Daily Report30.pdf,6.0,13.0,8,1,5



2026-08-25
  [HIGH    ] R2      NIGHT: rig ran 10.0 h inside a 8.25 h window (19:45–04:00). Impossible as stated — either the shift times or the hour meter is wrong.
  [HIGH    ] R6      Day ends at metre 261 but night starts at 231. The two shifts do not join up.
  [CRITICAL] R8      The WTD table books 13.0 m for BD25 but the shift log totals 30.0 m. The WTD table is what feeds the invoice — this is a billing exposure, not a typo.

2026-08-26
  [HIGH    ] R2      NIGHT: rig ran 10.0 h inside a 8.25 h window (19:45–04:00). Impossible as stated — either the shift times or the hour meter is wrong.
  [HIGH    ] R5      Counter reset to 0 (new hole) but Hole ID is still GPDD 048. The finished hole's final depth and the new hole ID are lost.
  [HIGH    ] R7      5.0 rig hours unaccounted between the previous report (ended 1756.0) and this one (starts 1761.0).
  [CRITICAL] R8      The WTD table books 34.0 m for BD26 but the shift log totals 33.0 m. The WTD table is what feeds the invoice —

---

## What to do with the output

1. **Any `CRITICAL` flag stops the invoice.** `MONEY-1` and `R8` both mean the
   metres that generate revenue disagree with the metres the client supervisor
   signed for. Resolve before anything goes to Shanta.
2. **`R2` and `R7` go to Hamza the same day**, while he can still remember what the
   hour meter actually read. A day later it is a guess.
3. **Read the update before the CEO does.** For the first month at least. The model
   writes what the data says, and the data currently says uncomfortable things.

## What this still cannot do

- **Machine status.** There is no downtime log, fault record, service history or
  hours-to-next-service anywhere on the sheet. Until those fields exist, the CEO's
  first instruction — *ensure the machine is up and running at all times* — cannot
  be reported on. The notebook can only tell you that.
- **Consumables and spares.** Same. Bit stock, rods, core barrels, mud, water,
  reorder levels, outstanding orders — none of it is on the daily report. Two
  stoppages in the last week of August prove the point: roughly three hours on the
  29th replacing a worn drill bit, and six hours on the 30th with the rig on standby
  waiting for water. **Nine hours, and the only record of either is a sentence Hamza
  typed into a free-text box.** Rule R15 catches the sentence; nothing totals the
  hours across the month, and nothing establishes whether standby is billable.
- **Cost per metre.** The `Fuel Liters` box records refuelling volume, not
  consumption — 5.02 L/m month-to-date against 11.93 L/m from a single daily sheet.
  Do not present the daily figure as consumption.

**Get the `.xlsx`.** Everything above gets more reliable and R8 stops depending on
text extraction.